<img style="float: left; margin: 30px 15px 15px 15px;" src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="500" height="250" /> 
    
    
# <font color='navy'> Credit Model

<font color='black'>

- Ivanna herrera Ibarra
- Ana Sofía Hinojosa Bale
- Luis Fernando Márquez Bañuelos

## <font color='cornflowerblue'> Packages

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

## <font color='cornflowerblue'> Data

In [2]:
# Safety measure for the dataset
data = pd.read_csv('hmeq.csv')
data = data.copy()

# Null values on MORTDUE are less than 10% of the data, given that is used as EAD, we will drop those rows.
data = data.dropna(subset=['MORTDUE'])
data

,BAD,LOAN,MORTDUE,VALUE,JOB,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC
0,1,1100,25860.0,39025.0,Other,10.5,0.0,0.0,94.366667,1.0,9.0,NaN
1,1,1300,70053.0,68400.0,Other,7.0,0.0,2.0,121.833333,0.0,14.0,NaN
2,1,1500,13500.0,16700.0,Other,4.0,0.0,0.0,149.466667,1.0,10.0,NaN
4,0,1700,97800.0,112000.0,Office,3.0,0.0,0.0,93.333333,0.0,14.0,NaN
5,1,1700,30548.0,40320.0,Other,9.0,0.0,0.0,101.466002,1.0,8.0,37.113614
...,...,...,...,...,...,...,...,...,...,...,...,...
5955,0,88900,57264.0,90185.0,Other,16.0,0.0,0.0,221.808718,0.0,16.0,36.112347
5956,0,89000,54576.0,92937.0,Other,16.0,0.0,0.0,208.692070,0.0,15.0,35.859971
5957,0,89200,54045.0,92924.0,Other,15.0,0.0,0.0,212.279697,0.0,15.0,35.556590
5958,0,89800,50370.0,91861.0,Other,14.0,0.0,0.0,213.892709,0.0,16.0,34.340882


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5442 entries, 0 to 5959
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   BAD      5442 non-null   int64  
 1   LOAN     5442 non-null   int64  
 2   MORTDUE  5442 non-null   float64
 3   VALUE    5357 non-null   float64
 4   JOB      5261 non-null   object 
 5   YOJ      5123 non-null   float64
 6   DEROG    4895 non-null   float64
 7   DELINQ   5041 non-null   float64
 8   CLAGE    5239 non-null   float64
 9   NINQ     5094 non-null   float64
 10  CLNO     5301 non-null   float64
 11  DEBTINC  4291 non-null   float64
dtypes: float64(9), int64(2), object(1)
memory usage: 552.7+ KB


In [4]:
data.describe()

,BAD,LOAN,MORTDUE,VALUE,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC
count,5442.000000,5442.000000,5442.000000,5357.000000,5123.000000,4895.000000,5041.000000,5239.000000,5094.000000,5301.000000,4291.000000
mean,0.199008,18646.361632,73760.817200,104998.161042,8.947316,0.245965,0.456854,179.170532,1.206125,22.000943,34.302520
std,0.399291,11087.763811,44457.609458,54141.433143,7.540593,0.818395,1.137638,84.203961,1.720253,10.002952,8.338714
min,0.000000,1100.000000,2063.000000,8000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.524499
25%,0.000000,11300.000000,46276.000000,69076.000000,3.000000,0.000000,0.000000,116.700000,0.000000,15.000000,29.669246
50%,0.000000,16500.000000,65019.000000,92179.000000,7.000000,0.000000,0.000000,173.575105,1.000000,21.000000,35.248898
75%,0.000000,23200.000000,91488.000000,123114.000000,13.000000,0.000000,0.000000,229.423877,2.000000,27.000000,39.212371
max,1.000000,89900.000000,399550.000000,512650.000000,41.000000,10.000000,15.000000,1168.233561,13.000000,71.000000,203.312149


## <font color='cornflowerblue'> Data Cleaning

In [5]:
# We just replace Nans on JOB with 'Other', given that it is a categorical variable and the missingness is not significant.
data['JOB'] = data['JOB'].fillna('Other')

In [6]:
# Preprocess categorical variables using one-hot encoding
categorical_cols = data.select_dtypes(include=['object']).columns
data = pd.get_dummies(data, columns=categorical_cols)
data

,BAD,LOAN,MORTDUE,VALUE,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,JOB_Mgr,JOB_Office,JOB_Other,JOB_ProfExe,JOB_Sales,JOB_Self
0,1,1100,25860.0,39025.0,10.5,0.0,0.0,94.366667,1.0,9.0,NaN,False,False,True,False,False,False
1,1,1300,70053.0,68400.0,7.0,0.0,2.0,121.833333,0.0,14.0,NaN,False,False,True,False,False,False
2,1,1500,13500.0,16700.0,4.0,0.0,0.0,149.466667,1.0,10.0,NaN,False,False,True,False,False,False
4,0,1700,97800.0,112000.0,3.0,0.0,0.0,93.333333,0.0,14.0,NaN,False,True,False,False,False,False
5,1,1700,30548.0,40320.0,9.0,0.0,0.0,101.466002,1.0,8.0,37.113614,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5955,0,88900,57264.0,90185.0,16.0,0.0,0.0,221.808718,0.0,16.0,36.112347,False,False,True,False,False,False
5956,0,89000,54576.0,92937.0,16.0,0.0,0.0,208.692070,0.0,15.0,35.859971,False,False,True,False,False,False
5957,0,89200,54045.0,92924.0,15.0,0.0,0.0,212.279697,0.0,15.0,35.556590,False,False,True,False,False,False
5958,0,89800,50370.0,91861.0,14.0,0.0,0.0,213.892709,0.0,16.0,34.340882,False,False,True,False,False,False


In [7]:
# For columns with integer values that tend to have an average of 0 or very low we use median imputation
# given its robustness to outliers, for the rest of the columns we will use KNN imputation.
data['DEROG'] = data['DEROG'].fillna(data['DEROG'].median())
data['DELINQ'] = data['DELINQ'].fillna(data['DELINQ'].median())
data['NINQ'] = data['NINQ'].fillna(data['NINQ'].median())

In [8]:
# Select only the numerical columns from the dataset
numerical_cols = data.select_dtypes(include=['float64', 'int64']).columns
X = data[numerical_cols].copy()

# Scaling data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# KNN Imputer
imputer = KNNImputer(n_neighbors=5, weights='distance')

# Impute missing values
X_imputed_scaled = imputer.fit_transform(X_scaled)

# Convert back to original scale
X_imputed = scaler.inverse_transform(X_imputed_scaled)

# Put back into dataframe
data[numerical_cols] = X_imputed

In [9]:
# For the column DEROG we round to the nearest integer to preserve its original meaning, given that it represents the number of derogatory reports.
data['DEROG'] = data['DEROG'].round().astype(int)

In [10]:
#data.to_csv('hmeq_cleaned.csv', index=False)

In [11]:
data = pd.read_csv('hmeq_cleaned.csv')
data

,BAD,LOAN,MORTDUE,VALUE,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC,JOB_Mgr,JOB_Office,JOB_Other,JOB_ProfExe,JOB_Sales,JOB_Self
0,0.0,8200.0,10909.0,37286.0,3.000000,0,0.0,268.254602,0.0,12.0,22.232953,False,False,True,False,False,False
1,0.0,23600.0,82407.0,106431.0,11.129866,0,0.0,169.805271,0.0,24.0,39.488844,False,False,True,False,False,False
2,0.0,18400.0,48970.0,69559.0,20.000000,0,0.0,178.015833,0.0,23.0,39.707988,False,False,False,True,False,False
3,0.0,14900.0,66329.0,100824.0,9.000000,0,0.0,279.240751,0.0,11.0,36.198375,False,False,False,True,False,False
4,0.0,18900.0,44605.0,65489.0,15.000000,0,0.0,189.008456,0.0,23.0,38.571083,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5164,0.0,14800.0,48910.0,70039.0,8.000000,0,1.0,94.971596,3.0,48.0,30.920502,False,False,True,False,False,False
5165,1.0,10000.0,91000.0,110000.0,16.000000,1,3.0,123.766667,2.0,22.0,30.105688,False,False,True,False,False,False
5166,0.0,29200.0,99600.0,143000.0,15.000000,0,0.0,232.333333,1.0,40.0,41.572755,False,False,True,False,False,False
5167,0.0,23800.0,45279.0,68939.0,6.000000,0,0.0,77.228822,1.0,19.0,35.521487,False,False,True,False,False,False


In [12]:
# Nivel de apalancamiento total
data['TOTAL_DEBT'] = data['LOAN'] + data['MORTDUE']

# Loan-to-Value: qué tan apalancado está el crédito vs. el inmueble
data['LTV'] = data['LOAN'] / data['VALUE']

# Combined LTV: suma deuda nueva + deuda hipotecaria existente sobre el valor
data['CLTV'] = data['TOTAL_DEBT'] / data['VALUE']

# Equity del propietario: qué tanto del inmueble es "suyo"
data['HOME_EQUITY'] = data['VALUE'] - data['MORTDUE']

# Equity ratio
data['EQUITY_RATIO'] = data['HOME_EQUITY'] / data['VALUE']

# Ratio de líneas delinquentes sobre total
data['DELINQ_RATIO'] = data['DELINQ'] / (data['CLNO'] + 1)  # +1 evita división por 0

# Equity negativo (underwater): deuda > valor del inmueble
data['UNDERWATER'] = (data['TOTAL_DEBT'] > data['VALUE']).astype(int)

data

,BAD,LOAN,MORTDUE,VALUE,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,...,JOB_ProfExe,JOB_Sales,JOB_Self,TOTAL_DEBT,LTV,CLTV,HOME_EQUITY,EQUITY_RATIO,DELINQ_RATIO,UNDERWATER
0,0.0,8200.0,10909.0,37286.0,3.000000,0,0.0,268.254602,0.0,12.0,...,False,False,False,19109.0,0.219922,0.512498,26377.0,0.707424,0.000000,0
1,0.0,23600.0,82407.0,106431.0,11.129866,0,0.0,169.805271,0.0,24.0,...,False,False,False,106007.0,0.221740,0.996016,24024.0,0.225724,0.000000,0
2,0.0,18400.0,48970.0,69559.0,20.000000,0,0.0,178.015833,0.0,23.0,...,True,False,False,67370.0,0.264524,0.968530,20589.0,0.295993,0.000000,0
3,0.0,14900.0,66329.0,100824.0,9.000000,0,0.0,279.240751,0.0,11.0,...,True,False,False,81229.0,0.147782,0.805651,34495.0,0.342131,0.000000,0
4,0.0,18900.0,44605.0,65489.0,15.000000,0,0.0,189.008456,0.0,23.0,...,True,False,False,63505.0,0.288598,0.969705,20884.0,0.318893,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5164,0.0,14800.0,48910.0,70039.0,8.000000,0,1.0,94.971596,3.0,48.0,...,False,False,False,63710.0,0.211311,0.909636,21129.0,0.301675,0.020408,0
5165,1.0,10000.0,91000.0,110000.0,16.000000,1,3.0,123.766667,2.0,22.0,...,False,False,False,101000.0,0.090909,0.918182,19000.0,0.172727,0.130435,0
5166,0.0,29200.0,99600.0,143000.0,15.000000,0,0.0,232.333333,1.0,40.0,...,False,False,False,128800.0,0.204196,0.900699,43400.0,0.303497,0.000000,0
5167,0.0,23800.0,45279.0,68939.0,6.000000,0,0.0,77.228822,1.0,19.0,...,False,False,False,69079.0,0.345233,1.002031,23660.0,0.343202,0.000000,1
